# Zadanie 3.1b: dwuprzedziałowy surogat ODE (`ODE_2`)

## Sformułowanie modelu

Rozdzielamy całkowitą masę guza $M(t) = M_{\text{in}}(t) + M_{\text{out}}(t)$ na dwa przedziały:
- $M_{\text{in}}(t) = \int_\Omega R(x)\,u(x, t)\,dx$ — masa w polu wiązki,
- $M_{\text{out}}(t) = \int_\Omega (1 - R(x))\,u(x, t)\,dx$ — masa poza polem wiązki.

Dawka promieniowania działa wyłącznie na $M_{\text{in}}$, a dyfuzja wymienia komórki przez brzeg
wiązki z tempem $\gamma = \gamma_0 D$ zależnym od współczynnika dyfuzji $D$:
$$\frac{dM_{\text{in}}}{dt} = \rho M_{\text{in}}\!\left(1 - \frac{M}{K_{\text{eff}}}\right) - \beta U(t)\,M_{\text{in}} + \gamma\big(c\,M_{\text{out}} - (1 - c)\,M_{\text{in}}\big),$$
$$\frac{dM_{\text{out}}}{dt} = \rho M_{\text{out}}\!\left(1 - \frac{M}{K_{\text{eff}}}\right) - \gamma\big(c\,M_{\text{out}} - (1 - c)\,M_{\text{in}}\big),$$
gdzie $c = M_{\text{in}}(0)/M(0)$ to początkowy udział guza w wiązce.

Postać członu wymiany ma trzy istotne własności:
1. Przy jednorodnej gęstości początkowej ($M_{\text{in}}/M = c$) strumień dyfuzyjny wynosi 0 — nie ma sztucznego przecieku przed podaniem pierwszej frakcji.
2. Sama dyfuzja ($\rho = \beta = 0$) ściśle zachowuje całkowitą masę $M_{\text{in}} + M_{\text{out}}$.
3. W granicy bardzo szybkiego mieszania ($\gamma \to \infty$) utrzymuje się $M_{\text{in}} = c M$ i układ redukuje się dokładnie do `ODE_1`; przy $c = 1$ oba modele są tożsame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, beam, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.ode import ODE1Params, ODE2Params, initial_beam_fraction, simulate_ode1, simulate_ode2
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
u0 = initial_condition(x)
M0 = float(tumor_mass(u0, x))
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)

# Scenariusz „pole przesunięte” (s = 6, r = 8, beta = 1.5)
p_shift = Params(beta=1.5, s=6.0, r=8.0)
R_shift = beam(x, p_shift)
c0 = initial_beam_fraction(x, u0, p_shift)
m_in0 = c0 * M0
m_out0 = (1.0 - c0) * M0

# Ręcznie dobrane bazowe K_eff i gamma_0 (szczegółowe strojenie w #20)
K_EFF = 30.0
GAMMA0 = 1.2

u_pde = simulate(p_shift, schedule, x, t, method="cn")
M_pde = tumor_mass(u_pde, x)
Min_pde = tumor_mass(u_pde * R_shift, x)
Mout_pde = tumor_mass(u_pde * (1.0 - R_shift), x)

M_ode1 = simulate_ode1(M0, ODE1Params(rho=p_shift.rho, K_eff=K_EFF, beta=p_shift.beta, c=c0), schedule, t)
Min_ode2, Mout_ode2 = simulate_ode2(
    m_in0, m_out0, ODE2Params(rho=p_shift.rho, K_eff=K_EFF, beta=p_shift.beta, c=c0, gamma=GAMMA0 * p_shift.D),
    schedule, t,
)
M_ode2 = Min_ode2 + Mout_ode2

## Wpływ dyfuzji $D$ ($\gamma = \gamma_0 D$) oraz porównanie metryk na „polu przesuniętym”

In [ ]:
rows = []
for label, M_cur in [("PDE (CN)", M_pde), ("ODE_1", M_ode1), ("ODE_2", M_ode2)]:
    mmin, tmin = min_mass(t, M_cur)
    treg = time_to_regrowth(t, M_cur)
    rel_l2 = 0.0 if label == "PDE (CN)" else float(np.linalg.norm(M_cur - M_pde) / np.linalg.norm(M_pde))
    rows.append({
        "model": label,
        "D": p_shift.D,
        "M_min": round(mmin, 3),
        "t_min": round(tmin, 2),
        "M_end": round(float(M_cur[-1]), 2),
        "t_odrostu": round(treg, 2) if np.isfinite(treg) else np.nan,
        "blad_L2_wzgl": round(rel_l2, 3),
    })

# Sprawdzenie skalowania gamma = gamma_0 * D dla różnych wartości dyfuzji D
D_sweep = [0.0, 0.25, 1.0, 3.0]
diff_curves = {}
for D_val in D_sweep:
    p_d = Params(D=D_val, beta=1.5, s=6.0, r=8.0)
    u_d = simulate(p_d, schedule, x, t, method="cn")
    M_d_pde = tumor_mass(u_d, x)
    mi_d, mo_d = simulate_ode2(
        m_in0, m_out0, ODE2Params(rho=p_d.rho, K_eff=K_EFF, beta=p_d.beta, c=c0, gamma=GAMMA0 * D_val),
        schedule, t,
    )
    M_d_ode2 = mi_d + mo_d
    diff_curves[D_val] = (M_d_pde, M_d_ode2)
    if D_val != 1.0:
        rows.append({
            "model": f"ODE_2 (D={D_val:g})",
            "D": D_val,
            "M_min": round(min_mass(t, M_d_ode2)[0], 3),
            "t_min": round(min_mass(t, M_d_ode2)[1], 2),
            "M_end": round(float(M_d_ode2[-1]), 2),
            "t_odrostu": round(time_to_regrowth(t, M_d_ode2), 2),
            "blad_L2_wzgl": round(float(np.linalg.norm(M_d_ode2 - M_d_pde) / np.linalg.norm(M_d_pde)), 3),
        })

df = pd.DataFrame(rows)
df.to_csv(TABLES / "zad3_01b_ode2.csv", index=False)
print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2))

# 1. Całkowita masa M(t)
ax = axes[0]
ax.plot(t, M_pde, color="black", lw=2.2, label="PDE (CN)")
ax.plot(t, M_ode1, color="tab:blue", ls="--", lw=1.8, label="ODE_1")
ax.plot(t, M_ode2, color="tab:orange", lw=2.0, label="ODE_2")
for ti in schedule.times:
    ax.axvline(ti, color="tab:red", alpha=0.14, lw=0.8)
ax.axhline(M0, color="gray", ls=":", lw=0.9)
ax.set_title("Całkowita masa $M(t)$: pole przesunięte")
ax.set_xlabel("czas $t$")
ax.set_ylabel("masa guza")
ax.legend(frameon=False)
ax.grid(alpha=0.25)

# 2. Podział na M_in i M_out
ax = axes[1]
ax.plot(t, Min_pde, color="tab:red", lw=2.0, label="PDE $M_{\\mathrm{in}}$")
ax.plot(t, Min_ode2, color="tab:red", ls="--", lw=1.8, label="ODE_2 $M_{\\mathrm{in}}$")
ax.plot(t, Mout_pde, color="tab:green", lw=2.0, label="PDE $M_{\\mathrm{out}}$")
ax.plot(t, Mout_ode2, color="tab:green", ls="--", lw=1.8, label="ODE_2 $M_{\\mathrm{out}}$")
ax.set_title("Przedziały $M_{\\mathrm{in}}$ (w wiązce) i $M_{\\mathrm{out}}$ (poza)")
ax.set_xlabel("czas $t$")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25)

# 3. Zależność od dyfuzji D
ax = axes[2]
colors = ["tab:purple", "tab:blue", "tab:orange", "tab:red"]
for col, D_val in zip(colors, D_sweep):
    Mp, Mo = diff_curves[D_val]
    ax.plot(t, Mp, color=col, lw=1.8, label=f"PDE $D={D_val:g}$")
    ax.plot(t, Mo, color=col, ls="--", lw=1.5)
ax.set_title("Wpływ dyfuzji ($\\gamma = \\gamma_0 D$): ciągła PDE, przerywana ODE_2")
ax.set_xlabel("czas $t$")
ax.legend(frameon=False, fontsize=8.5)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIGURES / "zad3_01b_ode2.png", dpi=150, bbox_inches="tight")
plt.close(fig)

## Wnioski

- Na scenariuszu **„pole przesunięte”** ($s=6, r=8$) `ODE_2` odtwarza mechanizm odrostu z nienapromienionego marginesu: podczas frakcji $M_{\text{in}}$ spada niemal do zera, ale $M_{\text{out}}$ przeżywa i po zakończeniu leczenia ponownie zasiedla strefę napromienioną (poprzez człon dyfuzyjny $\gamma$).
- Względny błąd $L^2$ spada wielokrotnie względem `ODE_1`, a czas powrotu do masy wyjściowej ($t_{\text{odrostu}}$) trafia bardzo blisko PDE (w przeciwieństwie do `ODE_1`, które w ogóle nie przewiduje odrostu przed $t = 10$).
- Skalowanie $\gamma = \gamma_0 D$ oddaje kierunek wpływu dyfuzji: większe $D$ zwiększa napływ komórek z marginesu do wiązki podczas frakcji (silniejsze wyczyszczanie w trakcie leczenia, niższe minimum masy).